**Avvio locale e Colab:** preparare la cartella completa seguendo il [README del laboratorio](README.md). Conservare gli identificativi delle celle e salvare gli output nel proprio runtime.

## C01 · PyTorch Studio
**Computer Vision · Lezione 10**

Quattro laboratori: dati, batch, aggiornamento, valutazione. Tre blocchi di codice da completare (C08, C12, C21). Conservare gli ID delle celle. Prima dell’esecuzione installare le dipendenze indicate nel README. Il test riusato da L9 è una replica didattica, non una nuova verifica indipendente.

### Lessico del percorso

- **Batch:** gruppo di esempi elaborati insieme. $B$ è la numerosità del batch corrente; $C=10$ è il numero delle classi. Nell'ultimo batch $B$ può essere più piccolo della dimensione nominale.
- **Aggiornamento o step:** una modifica dei parametri eseguita dall'ottimizzatore. Nel ciclo base di questo notebook c'è un `optimizer.step()` per batch di training; elaborare un batch di validation o test non aggiorna i pesi.
- **Epoca:** un passaggio sui batch del training. Qui ogni epoca visita tutti i 1077 esempi, conserva l'ultimo batch e cambia soltanto l'ordine con lo shuffle. Il training ripete questo passaggio per le 30 epoche stabilite.
- **Checkpoint del modello:** copia indipendente dello stato selezionato sulla validation, con epoca e metrica. Si conserva durante il training e si ripristina prima della valutazione finale. Una semplice etichetta dell'epoca non conserva i pesi.
- **Report finale:** riepilogo del protocollo e dei risultati del modello ripristinato: metriche, conteggi, matrice di confusione, errori e limiti. Produrre il report non è un nuovo aggiornamento e il test non sceglie il checkpoint.

Nei materiali del corso si usa il **punto decimale** (0.5) e la **virgola seguita da uno spazio** fra componenti di vettori e argomenti, per esempio (1.2, 1, −0.3). I conteggi interi non usano separatori delle migliaia.

### Avvio in Google Colab

Usare un runtime Python con **acceleratore hardware Nessuno**: la baseline rimane su CPU, con un thread e zero worker. Caricare il notebook e anche lo ZIP completo `PyTorch_Studio_studenti.zip` nello **stesso runtime**. Prima di C03, eseguire una cella temporanea di preparazione:

Per caricare il pacchetto completo e scegliere la cartella corrente seguire le istruzioni Colab nel README di questo laboratorio. Le celle scientifiche Cxx rimangono invariate.

Controllare prima dell’esecuzione gli import `torch`, `numpy`, `matplotlib`. Se una libreria manca, installare una versione compatibile con il runtime mediante `%pip install torch numpy matplotlib`; se occorre un riavvio, ripetere la preparazione. Non serve JupyterLab in Colab. C03 registra le versioni effettive. Eliminare la cella temporanea e conservare i 40 ID C01–C40. Dopo un riavvio, preparare nuovamente la cartella e rieseguire tutti i blocchi necessari in ordine, con C08/C12/C21 completate: gli output visibili non ricreano le variabili.

Prima di scollegare, usare **File → Scarica → Scarica .ipynb** per salvare il notebook con output. Dopo C36 e dopo aver scritto `output_lezione10/diagnosi.md`, scaricare anche i file esportati con una cella temporanea:

```python
import shutil
from google.colab import files

shutil.make_archive('/content/output_lezione10', 'zip', root_dir=str(OUT))
files.download('/content/output_lezione10.zip')
```

Eliminare anche questa cella prima della consegna. Salvare il notebook non trasferisce automaticamente i file del runtime. Il percorso documentato è stato verificato localmente su CPU; non è stata eseguita una prova nel servizio Colab. Registrare versioni, tempi e risultati della propria esecuzione. I prerequisiti fra i blocchi sono indicati nelle celle. [Guida ufficiale Colab](https://research.google.com/colaboratory/faq.html).

## C02 · Preparazione
Eseguire C03–C05. I dati sono locali. Le figure mostrano immagini reali di Digits con intensità 0–16. Le statistiche sono stimate soltanto sul training. In caso di interruzione ripartire da C03, completare i tre blocchi e procedere in ordine. Il dispositivo di riferimento è CPU.

In [ ]:
# C03
from pathlib import Path
import sys, json, hashlib, time, copy, platform
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
# Funziona nella cartella studenti, docente o nella radice del pacchetto estratto.
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd()/"notebook"]
BASE = next(p for p in candidates if (p/"dati/digits_cv2026.npz").exists())
sys.path.insert(0, str(BASE/"supporto"))
from studio_utils import (metrics, evaluate, save_figure, plot_data, plot_scaling,
                         plot_history, plot_confusion, plot_errors, challenge, export_run)
OUT = Path.cwd()/"output_lezione10"; OUT.mkdir(exist_ok=True)
DATA = BASE/"dati/digits_cv2026.npz"
DATA_SHA = hashlib.sha256(DATA.read_bytes()).hexdigest()
assert DATA_SHA == "972401a5680cf0a1c5ef60295fb915f4efbd51d7a39b35c0cfa39d73cc33ca1e"
archive = np.load(DATA)
images, targets = archive["images"], archive["target"]
indices = {"train": archive["train_idx"], "validation": archive["val_idx"], "test": archive["test_idx"]}
SEED=20261015; BATCH=64; EPOCHS=30; LR=0.05
torch.set_num_threads(1); torch.use_deterministic_algorithms(True)
versions = {"python": platform.python_version(), "torch": str(torch.__version__), "numpy": np.__version__, "matplotlib": matplotlib.__version__}
print(versions, "device=cpu, threads=1, workers=0")
print("SHA-256 dati:", DATA_SHA)

## C04 · Il contratto dei dati
1797 immagini 8×8, 10 classi, split L9 1077/360/360. Lo split è per immagini: mancano gli ID degli scriventi. Il preprocessing usa media e deviazione standard per pixel del training (ddof=0). Lo split non cambia nello Studio.

**Notazione raccordata alla lezione 9.** Il vettore degli score prima della softmax è $s$; la matrice dei vettori di un batch è $S$, di forma $B\times C$. Nel codice `logits` indica questi stessi score, mentre $p$ indica le probabilità. Per un esempio standardizzato, $s=W\tilde{x}+b$. La standardizzazione, detta anche *z-score*, è $\tilde{x}_j=(x_j-\mu_j)/a_j$: $\mu_j$ è la media del training e la scala $a_j$ è la sua deviazione standard se positiva, altrimenti 1 per la coordinata costante. Il codice le conserva in `mean` e `scale`. $a_j$ non è uno score; la parola *z-score* non cambia il simbolo $s$ dell'output del classificatore.


In [ ]:
# C05
assert images.shape == (1797,8,8) and targets.shape == (1797,)
all_ids=np.concatenate(list(indices.values()))
assert len(np.unique(all_ids)) == 1797 and np.array_equal(np.sort(all_ids), np.arange(1797))
print({k: len(v) for k,v in indices.items()}, "intensità:", (images.min(),images.max()))
plot_data(images,targets,indices["train"],OUT)

## C06 · Lab 1 · Dataset
Leggere C07, completare C08, poi eseguire C09. Restituire una coppia `(immagine, target)` mantenendo l’associazione attraverso l’indice originale. Rispondere in C10. 

In [ ]:
# C07
flat = images.reshape(len(images),64).astype(np.float64)
mean = flat[indices["train"]].mean(axis=0)
std = flat[indices["train"]].std(axis=0, ddof=0)
scale = np.where(std == 0, 1.0, std)
assert np.isfinite(mean).all() and (scale > 0).all()
print("Coordinate costanti nel training:", np.flatnonzero(std==0).tolist())
plot_scaling(images,targets,indices["train"][0],mean,scale,OUT)

In [ ]:
# C08
class DigitsDataset(Dataset):
    def __init__(self, images, targets, ids, mean, scale):
        self.images, self.targets = images, targets
        self.ids = np.asarray(ids).copy()
        self.mean, self.scale = mean.copy(), scale.copy()
    def __len__(self):
        # TODO: numerosità dello split.
        raise NotImplementedError("Completa __len__")
    def __getitem__(self, i):
        # TODO: indice originale, standardizzazione fissa, forma e dtype.
        raise NotImplementedError("Completa __getitem__")
datasets = {k: DigitsDataset(images, targets, v, mean, scale) for k,v in indices.items()}

In [ ]:
# C09
for name,ds in datasets.items():
    x,y=ds[0]; original_id=int(ds.ids[0])
    assert len(ds)==len(indices[name])
    assert x.shape==(1,8,8) and x.dtype==torch.float32
    assert y.shape==() and y.dtype==torch.long and y.item()==int(targets[original_id])
    expected=(flat[original_id]-mean)/scale
    assert np.allclose(x.numpy().reshape(64),expected,atol=1e-6)
    print(name, "N=",len(ds),"ID=",original_id,"target=",y.item(),"forma=",tuple(x.shape))

## C10 · Risposte Lab 1
**N01.** Che cosa rappresenta un elemento del Dataset? Riporta forma, tipo, intensità originarie e significato del target. Come rimane associato all’immagine?

**N02.** Quali indici determinano media e scala? Perché non rifarle su validation e test? Quale limite rimane con split disgiunti?

*Scrivere qui le risposte.*

## C11 · Lab 2 · DataLoader
Completare C12. Eseguire C13–C15 e rispondere in C16. Il loader training è ricreato per ogni nuova corsa. `train_eval` è un loader diverso e sequenziale. 

In [ ]:
# C12
def make_loaders(datasets, seed=SEED, batch_size=BATCH):
    # TODO: generatore dedicato, training mescolato e loader eval sequenziali.
    # Conservare l'ultimo batch. Restituire (loader_train, dizionario_eval).
    raise NotImplementedError("Completa make_loaders")
train_loader, eval_loaders = make_loaders(datasets)

In [ ]:
# C13
batch_report={}
for name,ds in datasets.items():
    loader=make_loaders(datasets)[0] if name=="train" else eval_loaders[name]
    sizes=[len(y) for x,y in loader]
    assert sum(sizes)==len(ds)
    assert sizes==([64]*16+[53] if name=="train" else [64]*5+[40])
    batch_report[name]={"batch":len(sizes),"N":sum(sizes),"ultimo":sizes[-1]}
print(batch_report)
# Controllo del sampler: ogni indice locale compare una sola volta.
fresh=make_loaders(datasets)[0]
order=list(iter(fresh.sampler))
assert sorted(order)==list(range(len(datasets["train"])))
print("Nessuna omissione o ripetizione:",len(order))

In [ ]:
# C14
a=make_loaders(datasets)[0]; b=make_loaders(datasets)[0]
a1=list(iter(a.sampler)); a2=list(iter(a.sampler)); b1=list(iter(b.sampler))
assert a1==b1 and a1!=a2
print("Primi ID originali epoca 1:", indices["train"][a1[:12]].tolist())
print("Primi ID originali epoca 2:", indices["train"][a2[:12]].tolist())
print("Ripartenza identica:",a1==b1,"ordine successivo diverso:",a1!=a2)
# Questa ispezione consuma il generatore di a; non riusiamo a per il training.

In [ ]:
# C15
xb,yb=next(iter(eval_loaders["train"]))
flatten=nn.Flatten(start_dim=1)
assert flatten(xb).shape==(len(xb),64)
print("Input:",tuple(xb.shape),xb.dtype,"target:",tuple(yb.shape),yb.dtype)
print("Flatten:",tuple(flatten(xb).shape))

## C16 · Risposte Lab 2
**N03.** Scrivi la catena delle forme per il batch completo e l’ultimo batch di training. Quale asse conserva Flatten?

**N05.** Conta batch ed esempi. Come ricostruisci la loss globale se l’ultimo batch è più piccolo?

**N12.** Che cosa va reimpostato per una nuova corsa? Quali informazioni vanno registrate oltre al seme?

*Scrivere qui le risposte.*

## C17 · Salvataggio e controllo dello stato
Salvare notebook e risposte. Conservare il kernel. 

## C18 · Lab 3 · Training loop
Eseguire C19–C20, completare le tre righe centrali di C21, poi C22–C26. C23–C24 sono forniti per osservare una corsa completa. Prima capire l’effetto di un solo batch. 

**Leggere il ciclo prima di eseguirlo.** Un'epoca attraversa 16 batch di 64 immagini e uno di 53: $1077=16\cdot64+53$. Con un aggiornamento per batch, C24 compie 17 step per epoca e 510 nelle 30 epoche. Dopo ogni epoca misura la macro-F1 sull'intera validation e conserva una copia dello stato soltanto se il valore supera strettamente il migliore precedente. Un pareggio esatto conserva la prima epoca. La minima loss e l'accuracy test non cambiano questo criterio. Il budget fisso completa tutte le 30 epoche: selezionare il checkpoint migliore fra gli stati visitati non introduce early stopping.


In [ ]:
# C19
def new_model(seed=SEED):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Flatten(start_dim=1), nn.Linear(64,10)).cpu()
criterion=nn.CrossEntropyLoss()
model=new_model()
assert sum(p.numel() for p in model.parameters())==650
logits=model(xb)
assert logits.shape==(len(xb),10)
print(model,"Parametri:",sum(p.numel() for p in model.parameters()),"logits:",tuple(logits.shape))

In [ ]:
# C20
uniform_logits=torch.full((3,10),7.0)
uniform_targets=torch.tensor([0,3,9],dtype=torch.long)
uniform_loss=criterion(uniform_logits,uniform_targets).item()
assert abs(uniform_loss-np.log(10))<1e-6
print("Controllo analitico CE uniforme:",uniform_loss,"nat; ln(10)=",np.log(10))
print("Argmax logits == argmax softmax:",torch.equal(logits.argmax(1),logits.softmax(1).argmax(1)))

In [ ]:
# C21
def update_batch(model, optimizer, xb, yb):
    # TODO 1: rimuovere i gradienti del batch precedente.
    logits=model(xb)
    loss=criterion(logits,yb)
    # TODO 2: calcolare e accumulare i gradienti.
    # TODO 3: aggiornare i parametri con SGD.
    raise NotImplementedError("Completa zero_grad, backward e step")
    return float(loss.detach())

In [ ]:
# C22
probe=new_model(); probe.train()
opt=torch.optim.SGD(probe.parameters(),lr=LR,momentum=0,weight_decay=0)
before={k:v.clone() for k,v in probe.state_dict().items()}
probe_loss=update_batch(probe,opt,xb,yb)
update_norm=float(torch.linalg.vector_norm(torch.cat([(probe.state_dict()[k]-v).flatten() for k,v in before.items()])))
grad_norm=float(torch.linalg.vector_norm(torch.cat([p.grad.flatten() for p in probe.parameters()])))
assert np.isfinite(probe_loss) and np.isfinite(grad_norm) and grad_norm>0 and update_norm>0
assert np.isclose(update_norm,LR*grad_norm,rtol=1e-5)
print({"loss_batch":probe_loss,"norma_gradiente":grad_norm,"norma_update":update_norm})

In [ ]:
# C23
# La funzione evaluate, leggibile in supporto/studio_utils.py, implementa:
# model.eval(); with torch.no_grad(): forward, loss e conteggi.
# Non invoca backward o step. Accumula loss_batch * B_corrente.
# Somma la confusion matrix, poi ricava le metriche sulle 10 classi.
print("Funzione di valutazione fornita. Il codice viene discusso nel Lab 4.")

In [ ]:
# C24
def fit_run():
    start=time.perf_counter(); model=new_model()
    optimizer=torch.optim.SGD(model.parameters(),lr=LR,momentum=0,weight_decay=0)
    training, loaders=make_loaders(datasets)  # nuovo generatore
    history=[]; best_f1=-1.0; best_state=None; first_batches=[]
    for epoch in range(1,EPOCHS+1):
        model.train(); total=0.0; seen=0
        for batch_no,(x,y) in enumerate(training):
            if batch_no==0: first_batches.append(hashlib.sha256(x.numpy().tobytes()+y.numpy().tobytes()).hexdigest())
            batch_loss=update_batch(model,optimizer,x,y)
            total+=batch_loss*len(y); seen+=len(y)
        train_eval=evaluate(model,loaders["train"],criterion)
        validation=evaluate(model,loaders["validation"],criterion)
        row={"epoch":epoch,"online_loss":total/seen,
             "train_eval_loss":train_eval["loss"],"validation_loss":validation["loss"],
             "train_eval_accuracy":train_eval["accuracy"],"validation_accuracy":validation["accuracy"],
             "train_eval_macro_f1":train_eval["macro_f1"],"validation_macro_f1":validation["macro_f1"]}
        assert seen==1077 and all(np.isfinite(v) for v in row.values())
        history.append(row)
        if validation["macro_f1"]>best_f1:
            best_f1=validation["macro_f1"]; best_epoch=epoch
            best_state=copy.deepcopy(model.state_dict())
    return {"history":history,"state":best_state,"best_epoch":best_epoch,
            "best_f1":best_f1,"first_batches":first_batches,"seconds":time.perf_counter()-start}

In [ ]:
# C25
run=fit_run()
# Seconda corsa indipendente: modello, SGD e generatore nuovamente inizializzati.
repeat=fit_run()
repro={"history_exact":run["history"]==repeat["history"],
       "batch_hashes_exact":run["first_batches"]==repeat["first_batches"],
       "weights_exact":all(torch.equal(run["state"][k],repeat["state"][k]) for k in run["state"])}
assert all(repro.values())
print("Epoca selezionata:",run["best_epoch"],"macro-F1 validation:",run["best_f1"])
print("Secondi delle due corse (ambiente locale):",round(run["seconds"],3),round(repeat["seconds"],3))
print("Replica nello stesso ambiente:",repro)

In [ ]:
# C26
plot_history(run["history"],run["best_epoch"],OUT)
print("Le curve train_eval e validation usano lo stesso stato a fine epoca.")
print("La loss online è registrata a parte in history.csv.")

## C27 · Risposte Lab 3
**N04.** Quanto vale la cross-entropy di dieci logits uguali? Perché passiamo logits a CrossEntropyLoss? Perché il controllo uniforme non predice la prima loss casuale?

**N06.** Che cosa cambiano zero_grad, backward e step? Che cosa rimane dopo due backward senza reset a pesi fissi?

*Scrivere qui le risposte.*

## C28 · Lab 4 · Validation e checkpoint
Leggere la funzione evaluate nel file supporto. Eseguire C29–C31 e rispondere in C32. Il test è consultato soltanto dopo il completamento della selezione su validation. Il suo riuso da L9 resta una replica didattica. 

**Che cosa significa conservare una copia prima del report.** C24 usa `copy.deepcopy(model.state_dict())` al miglioramento della validation, così il training successivo non modifica i tensori conservati. Una semplice assegnazione di `state_dict` mantiene riferimenti e non congela lo stato. C29 crea il modello `selected`, ricarica la copia con `load_state_dict` e solo dopo calcola le metriche del test. Ripristinare la copia non significa ripetere il training fino all'epoca scelta. Il checkpoint esportato in C36 consente inferenza con architettura e preprocessing coerenti; non comprende tutti gli stati dell'ottimizzatore e dei generatori richiesti per riprendere esattamente il training.


In [ ]:
# C29
selected=new_model(); selected.load_state_dict(run["state"])
selected_validation=evaluate(selected,eval_loaders["validation"],criterion)
assert selected_validation["macro_f1"]==run["best_f1"]
final_metrics=evaluate(selected,eval_loaders["test"],criterion)
print("Checkpoint ripristinato, epoca",run["best_epoch"])
print({k:v for k,v in final_metrics.items() if k not in ["confusion","pred","target","logits"]})

In [ ]:
# C30
plot_confusion(final_metrics["confusion"],OUT)
plot_errors(images,targets,indices["test"],final_metrics["pred"],OUT)

In [ ]:
# C31
before={k:v.clone() for k,v in selected.state_dict().items()}
a=evaluate(selected,eval_loaders["validation"],criterion)
b=evaluate(selected,DataLoader(datasets["validation"],batch_size=37,shuffle=False),criterion)
unchanged=all(torch.equal(before[k],selected.state_dict()[k]) for k in before)
assert unchanged and a["N"]==b["N"]==360
assert np.array_equal(a["pred"],b["pred"])
assert np.isclose(a["loss"],b["loss"],atol=1e-6)
print("Stato invariato:",unchanged,"loss B64:",a["loss"],"loss B37:",b["loss"])

## C32 · Risposte Lab 4
**N07.** In che cosa differiscono eval e no_grad? Quale controllo mostra che i pesi restano invariati?

**N08.** Perché usare una copia indipendente di state_dict?

**N09.** Quale criterio sceglie il checkpoint? Se la loss minima è a un’altra epoca, che cosa fai?

**N10.** Scrivi una frase corretta sul test L9 riusato.

**N11.** Descrivi una curva o un errore osservato, proponi un’ipotesi e un controllo.

*Scrivere qui le risposte.*

## C33 · Challenge · una causa, un controllo
Scegliere una prova: A, accumulo di gradienti a pesi fissi; B, media dei batch con taglie diverse; C, alias contro copia. Scrivere la previsione prima di eseguire C34. Le prove usano copie private del modello, non alterano selected.

**Previsione:** …

In [ ]:
# C34
CHOICE="A"  # una fra A, B e C
challenge_result=challenge(CHOICE,selected,datasets["validation"],criterion,LR)
print(json.dumps(challenge_result,indent=2))

## C35 · Diagnosi · massimo 150 parole
Il contratto violato è … Il controllo mostra … La correzione è … Da questo controllo non posso concludere …

Collegare la risposta all’output della propria prova. Una differenza di accuracy non è richiesta per provare il meccanismo. Per la prova B distinguere i numeri reali dal caso analitico 1.1111 contro 1.1667.

In [ ]:
# C36
config={"dataset_sha256":DATA_SHA,"split":{k:len(v) for k,v in indices.items()},
        "seed":SEED,"batch_size":BATCH,"epochs":EPOCHS,"lr":LR,"momentum":0,"weight_decay":0,
        "model":"Flatten + Linear(64, 10)","parameters":650,"device":"cpu","threads":1,"workers":0,
        "selection":"max validation macro_f1; first epoch on exact tie","versions":versions,
        "test_role":"L9 previously observed test: didactic replication, not independent development evaluation"}
export_run(OUT,config,indices,mean,scale,run,final_metrics,repro,challenge_result)
# Verifica del checkpoint destinato all'inferenza, non al resume esatto del training.
checkpoint=torch.load(OUT/"best_checkpoint.pt",map_location="cpu",weights_only=True)
reloaded=new_model(); reloaded.load_state_dict(checkpoint["state_dict"])
again=evaluate(reloaded,eval_loaders["test"],criterion)
assert np.array_equal(again["pred"],final_metrics["pred"])
assert np.allclose(again["logits"],final_metrics["logits"],atol=0,rtol=0)
print("Ricaricamento: stessi logits e predizioni. File salvati in",OUT.resolve())
print(sorted(p.name for p in OUT.iterdir()))

## C37 · Discussione conclusiva · E01–E05
1. Dataset e DataLoader: quali responsabilità?
2. Quali operazioni azzerano, accumulano e usano i gradienti?
3. Come aggreghi loss di batch diseguali?
4. Come scegli e conservi il checkpoint?
5. Quale limite rimane con seed fissato e test L9 riusato?

*Risposte individuali, prima del confronto finale.*

## C38 · Raccordo al 19 ottobre
La trattazione delle derivate spiega come il grafo produce le derivate e perché un aggiornamento può funzionare. Oggi conserviamo il contratto osservabile. Worker, device e nuovi dataset richiedono verifiche ulteriori; costituiscono approfondimenti distinti.

## C39 · Fonti
Documentazione ufficiale PyTorch: [Dataset/DataLoader](https://docs.pytorch.org/docs/2.14/data.html), [CrossEntropyLoss](https://docs.pytorch.org/docs/2.14/generated/torch.nn.CrossEntropyLoss.html), [autograd](https://docs.pytorch.org/docs/2.14/notes/autograd.html), [ciclo di ottimizzazione](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html), [salvataggio](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html), [riproducibilità](https://docs.pytorch.org/docs/2.14/notes/randomness.html). Versione realmente eseguita in C03 e config.json.

## C40 · Consegna · F01–F05
Consegnare notebook con N01–N12 e challenge; curve leggibili; metriche e matrice; configurazione e checkpoint; diagnosi di massimo 150 parole. Aggiungere le risposte al notebook e copiare la diagnosi in `output_lezione10/diagnosi.md`. C36 esporta i dati tecnici. Il checkpoint include preprocessing e indici, ma non tutti gli stati necessari per riprendere esattamente il training.